# 01 — Model (Task 2 Datavidia 11.0, Tim Perintis)

**Alur:** citra → preprocessing → DINOv2 → clustering kejadian → penamaan dari `rules.json` → label spreading → classifier → simpan model.

| Bagian | Sumber kode |
|---|---|
| 1. Preprocessing & ekstraksi | Hafizh (fungsi dikunci) |
| 2. Clustering & penamaan | Fahmi (`fahmi/B_kejadian.ipynb`, v1) |
| 3. Label spreading & classifier | Seno (`seno/C_model.ipynb`, run1) |

**Output:** `model/` → `model.joblib`, `pred_train.csv`, `events.parquet`, `event_names.csv`, `meta.json`.

Tidak ada label manusia per foto maupun per kelompok. Nama kelas hanya dari `rules.json`.

## 0. Library

In [ ]:
import json, os, platform
from datetime import datetime
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import joblib, sklearn

from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

## Pengaturan

In [ ]:
# --- Path (relatif terhadap folder notebook; sesuaikan dengan aturan pengumpulan) ---
DATA_DIR   = Path("Dataset/Fenomena")              # berisi <Id>/pre.jpg dan <Id>/post.jpg
SAMPLE_SUB = Path("Dataset/sample_submission.csv")
RULES_PATH = Path("rules.json")                    # salinan persis rules.json yang dikunci Fahmi
MODEL_DIR  = Path("model")

SEED = 42
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

# --- MODE_UJI: WAJIB False untuk versi yang dikumpulkan ---
MODE_UJI = True
FITUR_TERSIMPAN = Path("../hafizh/v2")                     # hanya dipakai saat MODE_UJI
PHYS_TERSIMPAN  = Path("../hafizh/v3/phys_v4.parquet")     # hanya dipakai saat MODE_UJI
B_ACUAN = Path("../fahmi/v1")                              # cek: hasil B di sini harus sama dengan B asli
C_ACUAN = Path("../seno/run1")                             # cek: prediksi di sini harus sama dengan run C
if MODE_UJI:
    SAMPLE_SUB = next(Path(p) for p in ["../../Task-2/sample_submission.csv", "../../sample_submission.csv",
                                        "../Dataset/sample_submission.csv", str(SAMPLE_SUB)] if Path(p).exists())

# --- B (sama dengan fahmi/v1/meta.json) ---
B_PCA, B_ROUND, B_K, B_KNN = 64, 4, 20, 15
B_MIN_WARNA = 15
IND_STRUKTUR = ["d_dark", "d_entropy", "delta_edge", "edge_pre",
                "top10_share", "radial_decay", "com_shift", "frac_chg_p15"]
IND_WARNA    = ["d_exg_v4", "water_rel_pre", "d_water", "d_bright_raw_v4"]

# --- C (sama dengan seno/run1/meta.json) ---
SUMBER_LABEL = "pusat_semua_cluster"     # atau "is_core"
CORE_FRAC = 0.5
LS_KNN, LS_ALPHA = 15, 0.2
FITUR_BLOK = ["color_pre", "color_post"]
C_PCA, LR_C = 128, 1.0

np.random.seed(SEED)
with open(RULES_PATH, encoding="utf-8") as f:
    RULES = json.load(f)
assert RULES["status"] == "dikunci", "rules.json belum dikunci"
print("MODE_UJI =", MODE_UJI, "| sample_submission:", SAMPLE_SUB, "| rules dikunci:", RULES["dikunci_pada"])

## 1. [A — Hafizh] Preprocessing & ekstraksi fitur (DIKUNCI)

Sel berikut diisi **salinan persis** fungsi preprocessing + ekstraksi yang dikunci Hafizh (sama di `01_model` dan `02_submission`).

Kontrak keluaran `ekstrak_semua(data_dir, ids)`:

| Kunci | Isi |
|---|---|
| `emb` | dict `{"color_pre", "color_post", ...}` → `np.ndarray (N, 1024) float32`, urutan = `ids` |
| `art` | DataFrame `artifact_and_phys_v3` (kolom `Id`, urutan = `ids`) |
| `patch` | DataFrame `patch_change_stats_v3` (kolom `Id`, urutan = `ids`) |
| `phys` | DataFrame `phys_v4` (kolom `Id`, urutan = `ids`) |

Selama `MODE_UJI = True`, fitur dimuat dari file tersimpan A supaya sisa notebook bisa diuji sebelum fungsi A siap.

In [ ]:
# ===== TEMPEL DI SINI: fungsi preprocessing + ekstraksi yang dikunci Hafizh =====
# def ekstrak_semua(data_dir, ids):
#     ...
#     return {"emb": emb, "art": art, "patch": patch, "phys": phys}
# ================================================================================

In [ ]:
sub = pd.read_csv(SAMPLE_SUB)
ids = sub["Id"].to_numpy()
N = len(ids)
assert sub["Id"].is_unique and sub.columns[1] == "Phenomenon"

if MODE_UJI:
    print("PERINGATAN MODE_UJI: fitur dimuat dari file tersimpan, BUKAN diekstrak dari citra")
    fitur = {
        "emb": {b: np.load(FITUR_TERSIMPAN / f"dino_{b}.npy") for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"})},
        "art": pd.read_parquet(FITUR_TERSIMPAN / "artifact_and_phys_v3.parquet"),
        "patch": pd.read_parquet(FITUR_TERSIMPAN / "patch_change_stats_v3.parquet"),
        "phys": pd.read_parquet(PHYS_TERSIMPAN),
    }
else:
    assert "ekstrak_semua" in globals(), "fungsi ekstrak_semua dari Hafizh belum ditempel di sel atas"
    fitur = ekstrak_semua(DATA_DIR, ids)

emb = fitur["emb"]
for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"}):
    assert emb[b].shape == (N, 1024) and not np.isnan(emb[b]).any(), f"embedding {b} bermasalah"

art, patch, phys = fitur["art"], fitur["patch"], fitur["phys"]
for nama_t, df in [("art", art), ("patch", patch), ("phys", phys)]:
    assert len(df) == N and (df["Id"].to_numpy() == ids).all(), f"urutan Id {nama_t} beda"

print("N =", N, "| blok embedding:", sorted(emb))

## 2. [B — Fahmi] Clustering kejadian & penamaan dari `rules.json`
Salinan kode `fahmi/B_kejadian.ipynb` (v1) tanpa gate dan plot. Kalau Fahmi membuat versi baru, salin ulang bagian ini.

In [ ]:
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)

def bangun_fitur(emb, blok):
    # fitur per foto: L2 per blok embedding, digabung (tanpa event_id)
    return np.hstack([l2(emb[b]) for b in blok])

# Ruang Z: L2 per blok -> gabung pre+post -> kurangi rata-rata -> PCA -> bulatkan
X_b = np.hstack([l2(emb["color_pre"]), l2(emb["color_post"])])
X_b = X_b - X_b.mean(axis=0)
Z = np.round(PCA(B_PCA, random_state=SEED).fit_transform(X_b), B_ROUND)

event_id = SpectralClustering(n_clusters=B_K, affinity="nearest_neighbors", n_neighbors=B_KNN,
                              assign_labels="cluster_qr", random_state=SEED).fit_predict(Z)
ukuran = pd.Series(event_id).value_counts().sort_index()
print("Ukuran cluster:", ukuran.to_dict())

In [ ]:
# Median indikator per cluster -> z-score antar cluster
sumber = {"d_dark": phys, "d_entropy": phys, "delta_edge": art, "edge_pre": art,
          "top10_share": patch, "radial_decay": patch, "com_shift": patch, "frac_chg_p15": patch,
          "d_exg_v4": phys, "water_rel_pre": phys, "d_water": phys, "d_bright_raw_v4": phys}
IND_SEMUA = IND_STRUKTUR + IND_WARNA
ind = pd.DataFrame({c: sumber[c][c].to_numpy() for c in IND_SEMUA})
ind["event_id"] = event_id

g = ind.groupby("event_id")
med = g[IND_SEMUA].median()
n_valid = g[IND_WARNA].count()
for c in IND_WARNA:
    med.loc[n_valid[c] < B_MIN_WARNA, c] = np.nan
zc = (med - med.mean()) / med.std(ddof=0)

# Skor per keluarga: rata-rata (tanda x z), NaN dilewati
s1 = pd.DataFrame(index=zc.index, columns=KELAS, dtype=float)
s2 = pd.DataFrame(index=zc.index, columns=KELAS, dtype=float)
for kls in KELAS:
    s1[kls] = pd.concat([t * zc[c] for c, t in RULES["keluarga_1_piksel"][kls].items()], axis=1).mean(axis=1)
    s2[kls] = pd.concat([t * zc[c] for c, t in RULES["keluarga_2_patch"][kls].items()], axis=1).mean(axis=1)

# Aturan NaN keluarga 1: cluster dengan NaN di salah satu kelas -> keluarga 2 saja
hanya_k2 = s1.isna().any(axis=1)
skor = (s1 + s2) / 2
skor[s1.isna() & s2.notna()] = s2
skor[s2.isna() & s1.notna()] = s1
skor.loc[hanya_k2] = s2.loc[hanya_k2]
assert not skor.isna().any().any()

urut = np.sort(skor.to_numpy(), axis=1)
margin = pd.Series(urut[:, -1] - urut[:, -2], index=skor.index)

# Kelas kosong: pindahkan cluster dengan pengorbanan terkecil
nama = skor.idxmax(axis=1)
while True:
    kosong = [k for k in KELAS if (nama == k).sum() == 0]
    if not kosong:
        break
    kandidat = [(skor.loc[e, nama[e]] - skor.loc[e, k], e, k)
                for e in nama.index if (nama == nama[e]).sum() > 1 for k in kosong]
    if not kandidat:
        break
    _, e, k = min(kandidat)
    nama[e] = k

names = pd.DataFrame({"Phenomenon": nama, "margin": margin.round(4),
                      "is_weak": margin < RULES["margin_lemah"], "n": ukuran})
names.index.name = "event_id"
nama_cluster = names["Phenomenon"].reindex(event_id).to_numpy()
print("Cluster lemah:", int(names["is_weak"].sum()), "dari", B_K)
names.groupby("Phenomenon")["n"].agg(["count", "sum"]).reindex(KELAS).fillna(0).astype(int)

In [ ]:
# Cek: hasil B di notebook ini sama dengan output B asli (jika foldernya ada)
if B_ACUAN.exists():
    ev_ref = pd.read_parquet(B_ACUAN / "events.parquet")
    nm_ref = pd.read_csv(B_ACUAN / "event_names.csv").set_index("event_id")
    assert (ev_ref["Id"].to_numpy() == ids).all()
    assert (ev_ref["event_id"].to_numpy() == event_id).all(), "event_id beda dengan B asli"
    assert (nm_ref["Phenomenon"] == names["Phenomenon"]).all(), "nama kelas beda dengan B asli"
    assert (nm_ref["is_weak"] == names["is_weak"]).all(), "is_weak beda dengan B asli"
    print("Hasil B sama 100% dengan", B_ACUAN)
else:
    print("B_ACUAN tidak ada, cek dilewati")

## 3. [C — Seno] Seed → label spreading → classifier
Seed: `CORE_FRAC` foto terdekat pusat cluster di ruang `Z`. Classifier hanya memakai embedding per foto (tanpa `event_id`).

In [ ]:
def pilih_seed(lewati_lemah):
    seed = np.zeros(N, dtype=bool)
    for e in names.index:
        if lewati_lemah and names.loc[e, "is_weak"]:
            continue
        idx = np.where(event_id == e)[0]
        jarak = np.linalg.norm(Z[idx] - Z[idx].mean(axis=0), axis=1)
        seed[idx[np.argsort(jarak, kind="stable")[:int(round(CORE_FRAC * len(idx)))]]] = True
    return seed

is_seed = pilih_seed(lewati_lemah=(SUMBER_LABEL == "is_core"))

y = np.full(N, -1)
y[is_seed] = pd.Categorical(nama_cluster[is_seed], categories=KELAS).codes
ls = LabelSpreading(kernel="knn", n_neighbors=LS_KNN, alpha=LS_ALPHA, max_iter=1000).fit(Z, y)
y_latih = np.array(KELAS)[ls.classes_][ls.label_distributions_.argmax(axis=1)]
print(f"{SUMBER_LABEL}: seed {int(is_seed.sum())} | kelas terisi {len(set(y_latih))}/6")

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(C_PCA, random_state=SEED)),
    ("clf", LogisticRegression(class_weight="balanced", C=LR_C, max_iter=5000)),
]).fit(bangun_fitur(emb, FITUR_BLOK), y_latih)

X_c = bangun_fitur(emb, FITUR_BLOK)
pred = model.predict(X_c)
proba = model.predict_proba(X_c)
pd.Series(pred).value_counts().reindex(KELAS).fillna(0).astype(int).to_frame("n_prediksi")

## 4. Simpan model

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
VERSI_LIB = {"python": platform.python_version(), "numpy": np.__version__,
             "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}

joblib.dump({"pipeline": model, "fitur_blok": FITUR_BLOK, "l2_per_blok": True,
             "kelas": [str(k) for k in model.classes_], "versi_lib": VERSI_LIB},
            MODEL_DIR / "model.joblib")

pred_train = pd.DataFrame({"Id": ids, "Phenomenon": pred, "label_latih": y_latih, "is_seed": is_seed})
for i, k in enumerate(model.classes_):
    pred_train[f"proba_{k}"] = proba[:, i].round(6)
pred_train.to_csv(MODEL_DIR / "pred_train.csv", index=False)
pd.DataFrame({"Id": ids, "event_id": event_id, "is_seed": is_seed}).to_parquet(MODEL_DIR / "events.parquet", index=False)
names.reset_index().to_csv(MODEL_DIR / "event_names.csv", index=False)

meta = {
    "tanggal": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "mode_uji": MODE_UJI,
    "rules": {"status": RULES["status"], "dikunci_pada": RULES["dikunci_pada"]},
    "b": {"pca": B_PCA, "round": B_ROUND, "K": B_K, "knn": B_KNN, "min_warna": B_MIN_WARNA},
    "c": {"sumber_label": SUMBER_LABEL, "core_frac": CORE_FRAC, "ls_knn": LS_KNN, "ls_alpha": LS_ALPHA,
          "fitur_blok": FITUR_BLOK, "c_pca": C_PCA, "lr_C": LR_C, "class_weight": "balanced"},
    "seed": SEED,
    "distribusi_prediksi": pd.Series(pred).value_counts().reindex(KELAS).fillna(0).astype(int).to_dict(),
    "versi_lib": VERSI_LIB,
}
with open(MODEL_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Disimpan ke", MODEL_DIR, ":", sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Cek: prediksi sama dengan run C (jika foldernya ada)
if C_ACUAN.exists():
    ref = pd.read_csv(C_ACUAN / "pred_train.csv")
    assert (ref["Id"].to_numpy() == ids).all()
    sama = (ref["Phenomenon"].to_numpy() == pred).mean()
    print(f"Sama dengan {C_ACUAN}: {sama:.4f}")
    assert sama == 1.0, "prediksi beda dengan run C"

# requirements.txt dari lingkungan yang menjalankan notebook ini (hanya versi final)
PAKET = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "joblib",
         "torch", "torchvision", "transformers", "timm", "opencv-python", "scikit-image", "Pillow"]
baris = []
for p in PAKET:
    try:
        baris.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        pass
if MODE_UJI:
    print("MODE_UJI: requirements.txt TIDAK ditulis. Versi saat ini:", baris)
else:
    Path("requirements.txt").write_text("\n".join(baris) + "\n", encoding="utf-8")
    print("requirements.txt ditulis:", baris)